In [1]:
!pip install rank-bm25 sentence-transformers groq scikit-learn

In [2]:
import os, getpass, pathlib
from dotenv import load_dotenv
from groq import Groq
import re
import json
import numpy as np
import pandas as pd
from typing import List, Dict

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from sklearn.metrics.pairwise import cosine_similarity

In [31]:
# ---- API key (secure) ----
import os
import getpass

try:
    # Try loading from local file (for your use)
    with open("groq_API_Key.txt", "r") as f:
        os.environ["GROQ_API_KEY"] = f.read().strip()
    print("Loaded API key from file")
except FileNotFoundError:
    # Fallback for evaluator / first-time run
    if not os.getenv("GROQ_API_KEY"):
        os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key: ")

# ---- Groq client (UNCHANGED LOGIC) ----
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

groq_client = None
if GROQ_API_KEY:
    from groq import Groq
    groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq configured:", bool(groq_client))

Loaded API key from file
Groq configured: True


In [4]:
corpus = [
    "Transformers use self-attention mechanisms to encode relationships between words in a sequence.",
    "Self-attention computes pairwise interactions between tokens to capture contextual meaning.",
    "The attention mechanism assigns weights to tokens based on relevance to a query.",
    
    "Gradient descent is an optimization algorithm used to minimize loss functions.",
    "Adam optimizer combines momentum and adaptive learning rates for efficient training.",
    "Backpropagation computes gradients of loss with respect to neural network weights.",
    
    "Overfitting occurs when a model memorizes training data and fails to generalize.",
    "Regularization techniques like dropout help prevent overfitting in neural networks.",
    
    "BERT is a transformer-based model pretrained using masked language modeling.",
    "Tokenization converts raw text into tokens for model input processing.",
    
    "TF-IDF stands for Term Frequency-Inverse Document Frequency and is used in information retrieval."
]

for i, doc in enumerate(corpus):
    print(f"{i:02d} | {doc}")

00 | Transformers use self-attention mechanisms to encode relationships between words in a sequence.
01 | Self-attention computes pairwise interactions between tokens to capture contextual meaning.
02 | The attention mechanism assigns weights to tokens based on relevance to a query.
03 | Gradient descent is an optimization algorithm used to minimize loss functions.
04 | Adam optimizer combines momentum and adaptive learning rates for efficient training.
05 | Backpropagation computes gradients of loss with respect to neural network weights.
06 | Overfitting occurs when a model memorizes training data and fails to generalize.
07 | Regularization techniques like dropout help prevent overfitting in neural networks.
08 | BERT is a transformer-based model pretrained using masked language modeling.
09 | Tokenization converts raw text into tokens for model input processing.
10 | TF-IDF stands for Term Frequency-Inverse Document Frequency and is used in information retrieval.


In [5]:
def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

def tokenize_for_bm25(text: str) -> List[str]:
    return normalize_text(text).split()

def rrf_score(rank: int, k: int = 60) -> float:
    return 1.0 / (k + rank)

def pretty_print_results(results: List[Dict], title: str = "Results"):
    print(f"\n{title}")
    print("-" * len(title))
    for i, item in enumerate(results, start=1):
        print(f"{i}. doc_id={item['doc_id']}")
        if 'rrf_score' in item:
            print(f"   rrf_score   = {item['rrf_score']:.6f}")
        if 'bm25_rank' in item:
            print(f"   bm25_rank   = {item['bm25_rank']}")
        if 'sbert_rank' in item:
            print(f"   sbert_rank  = {item['sbert_rank']}")
        if 'cross_score' in item:
            print(f"   cross_score = {item['cross_score']:.6f}")
        print(f"   text        = {item['text']}")

In [6]:
class HybridRetriever:
    def __init__(self, corpus: List[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        # BM25 index
        self.tokenized_corpus = [tokenize_for_bm25(doc) for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized_corpus)

        # SBERT index
        self.embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.doc_embeddings = self.embedder.encode(
            corpus,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        # ----- BM25 ranking -----
        bm25_scores = self.bm25.get_scores(tokenize_for_bm25(query))
        bm25_order = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(bm25_order)}

        # ----- SBERT ranking -----
        query_emb = self.embedder.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        dense_scores = cosine_similarity(query_emb, self.doc_embeddings)[0]
        sbert_order = np.argsort(dense_scores)[::-1]
        sbert_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(sbert_order)}

        # ----- RRF fusion -----
        fused = []
        for doc_id, text in enumerate(self.corpus):
            bm25_rank = bm25_ranks[doc_id]
            sbert_rank = sbert_ranks[doc_id]
            score = rrf_score(bm25_rank, self.k) + rrf_score(sbert_rank, self.k)
            fused.append({
                "doc_id": doc_id,
                "rrf_score": float(score),
                "bm25_rank": int(bm25_rank),
                "sbert_rank": int(sbert_rank),
                "text": text
            })

        fused = sorted(fused, key=lambda x: x["rrf_score"], reverse=True)
        return fused[:top_k]

In [7]:
class NaiveDenseRetriever:
    def __init__(self, corpus: List[str]):
        self.corpus = corpus
        self.embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        self.doc_embeddings = self.embedder.encode(
            corpus,
            convert_to_numpy=True,
            normalize_embeddings=True
        )

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        q_emb = self.embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
        scores = cosine_similarity(q_emb, self.doc_embeddings)[0]
        order = np.argsort(scores)[::-1][:top_k]
        return [
            {
                "doc_id": int(i),
                "score": float(scores[i]),
                "text": self.corpus[int(i)]
            }
            for i in order
        ]

In [8]:
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query: str, candidates: List[Dict], top_k: int = 3) -> List[Dict]:
    """
    Re-rank candidates using the ORIGINAL user query.
    """
    pairs = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)

    reranked = []
    for cand, score in zip(candidates, scores):
        item = cand.copy()
        item["cross_score"] = float(score)
        reranked.append(item)

    reranked = sorted(reranked, key=lambda x: x["cross_score"], reverse=True)
    return reranked[:top_k]

In [9]:
def generate_multi_queries(user_query: str, n: int = 3) -> List[str]:
    if groq_client is None:
        # Deterministic fallback if API key is missing
        return [
            user_query,
            f"technical explanation of {user_query}",
            f"machine learning concept: {user_query}",
            f"detailed academic definition of {user_query}"
        ][:n+1]

    prompt = f"""
You are helping with retrieval for a university knowledge assistant.

Generate exactly {n} alternative search queries for this user question.
The goal is retrieval improvement, so the queries should:
- preserve the same meaning
- use more technical vocabulary when useful
- be concise
- not answer the question

Return ONLY a JSON array of strings.

User question: {user_query}
""".strip()

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You generate retrieval-focused paraphrases and respond only in valid JSON."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0
    )

    text = response.choices[0].message.content.strip()
    try:
        parsed = json.loads(text)
        if isinstance(parsed, list):
            parsed = [q.strip() for q in parsed if isinstance(q, str) and q.strip()]
            return [user_query] + parsed[:n]
    except Exception:
        pass

    # Safe fallback parsing
    lines = [ln.strip("-•1234567890. ").strip() for ln in text.splitlines() if ln.strip()]
    lines = [ln for ln in lines if len(ln) > 3]
    return [user_query] + lines[:n]


In [10]:
hybrid = HybridRetriever(corpus)
naive_retriever = NaiveDenseRetriever(corpus)

def expanded_hybrid_retrieve(user_query: str, per_query_top_k: int = 5, final_pool_k: int = 8) -> List[Dict]:
    queries = generate_multi_queries(user_query, n=3)
    print("Expanded queries:")
    for q in queries:
        print(" -", q)

    pool = {}
    for q in queries:
        results = hybrid.retrieve(q, top_k=per_query_top_k)
        for item in results:
            doc_id = item["doc_id"]
            # Keep the best fused score if the doc appears multiple times
            if doc_id not in pool or item["rrf_score"] > pool[doc_id]["rrf_score"]:
                pool[doc_id] = item

    merged = sorted(pool.values(), key=lambda x: x["rrf_score"], reverse=True)
    return merged[:final_pool_k]

In [11]:
def generate_answer_with_context(user_query: str, contexts: List[Dict]) -> str:
    context_text = "\n".join([f"[Doc {i+1}] {c['text']}" for i, c in enumerate(contexts)])

    prompt = f"""
You are a helpful university AI/ML teaching assistant.

Answer the student's question using ONLY the provided context.
If the context is insufficient, say so clearly.
Keep the answer concise, clear, and technically correct.

Student question:
{user_query}

Retrieved context:
{context_text}
""".strip()

    if groq_client is not None:
        response = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "You answer clearly and accurately using retrieved context only."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2
        )
        return response.choices[0].message.content.strip()

    # Offline fallback
    return "No Groq API key found. Retrieval pipeline worked, but answer generation was skipped."


In [12]:
def advanced_rag(user_query: str) -> str:
    """
    Full pipeline:
    Query Expansion -> Hybrid Retrieval -> Re-Ranking -> LLM Generation
    """
    # Step 1: expanded retrieval
    candidates = expanded_hybrid_retrieve(user_query, per_query_top_k=5, final_pool_k=8)
    pretty_print_results(candidates, title="Hybrid candidates before re-ranking")

    # Step 2: cross-encoder re-ranking using ORIGINAL query
    top_docs = rerank(user_query, candidates, top_k=3)
    pretty_print_results(top_docs, title="Top documents after cross-encoder re-ranking")

    # Step 3: answer generation
    answer = generate_answer_with_context(user_query, top_docs)
    return answer

In [13]:
test_query = "how do transformers encode meaning?"
hybrid_results = hybrid.retrieve(test_query, top_k=5)
pretty_print_results(hybrid_results, title="Hybrid retrieval sanity check")

reranked_results = rerank(test_query, hybrid_results, top_k=3)
pretty_print_results(reranked_results, title="Re-ranked results sanity check")


Hybrid retrieval sanity check
-----------------------------
1. doc_id=0
   rrf_score   = 0.032787
   bm25_rank   = 1
   sbert_rank  = 1
   text        = Transformers use self-attention mechanisms to encode relationships between words in a sequence.
2. doc_id=9
   rrf_score   = 0.032002
   bm25_rank   = 2
   sbert_rank  = 3
   text        = Tokenization converts raw text into tokens for model input processing.
3. doc_id=8
   rrf_score   = 0.031754
   bm25_rank   = 4
   sbert_rank  = 2
   text        = BERT is a transformer-based model pretrained using masked language modeling.
4. doc_id=10
   rrf_score   = 0.030366
   bm25_rank   = 3
   sbert_rank  = 9
   text        = TF-IDF stands for Term Frequency-Inverse Document Frequency and is used in information retrieval.
5. doc_id=7
   rrf_score   = 0.030310
   bm25_rank   = 5
   sbert_rank  = 7
   text        = Regularization techniques like dropout help prevent overfitting in neural networks.

Re-ranked results sanity check
---------------

In [17]:
query = "how do transformers encode meaning?"
answer = advanced_rag(query)
print("\nFinal Answer:")
print(answer)

Expanded queries:
 - how do transformers encode meaning?
 - What are the semantic encoding mechanisms in transformer architectures?
 - How do transformer models represent linguistic meaning?
 - What role do self-attention mechanisms play in encoding meaning in transformers?

Hybrid candidates before re-ranking
-----------------------------------
1. doc_id=0
   rrf_score   = 0.032787
   bm25_rank   = 1
   sbert_rank  = 1
   text        = Transformers use self-attention mechanisms to encode relationships between words in a sequence.
2. doc_id=9
   rrf_score   = 0.032002
   bm25_rank   = 2
   sbert_rank  = 3
   text        = Tokenization converts raw text into tokens for model input processing.
3. doc_id=8
   rrf_score   = 0.032002
   bm25_rank   = 3
   sbert_rank  = 2
   text        = BERT is a transformer-based model pretrained using masked language modeling.
4. doc_id=2
   rrf_score   = 0.031754
   bm25_rank   = 2
   sbert_rank  = 4
   text        = The attention mechanism assigns weig

In [18]:
def naive_top_doc(query: str) -> str:
    return naive_retriever.retrieve(query, top_k=1)[0]["text"]

def advanced_top_doc(query: str) -> str:
    candidates = expanded_hybrid_retrieve(query, per_query_top_k=5, final_pool_k=8)
    reranked = rerank(query, candidates, top_k=1)
    return reranked[0]["text"]

test_queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what helps retrieval when query words do not match document words?"
]

rows = []
for q in test_queries:
    naive_doc = naive_top_doc(q)
    adv_doc = advanced_top_doc(q)
    rows.append({
        "Query": q,
        "Naive RAG Top Doc": naive_doc,
        "Advanced RAG Top Doc": adv_doc,
        "Are they different?": "Yes" if naive_doc != adv_doc else "No"
    })

comparison_df = pd.DataFrame(rows)
comparison_df

Expanded queries:
 - how do transformers encode meaning?
 - What are the semantic encoding mechanisms in transformer architectures?
 - How do transformer models represent linguistic meaning?
 - What is the process of meaning representation in transformer-based neural networks?
Expanded queries:
 - optimization techniques for training
 - machine learning optimization methods
 - neural network training optimization strategies
 - deep learning model optimization techniques
Expanded queries:
 - what helps retrieval when query words do not match document words?
 - What techniques improve recall when query terms do not match document terms?
 - How does semantic matching enhance information retrieval?
 - What role does synonym expansion play in query term mismatch scenarios?


,Query,Naive RAG Top Doc,Advanced RAG Top Doc,Are they different?
0,how do transformers encode meaning?,Transformers use self-attention mechanisms to ...,Transformers use self-attention mechanisms to ...,No
1,optimization techniques for training,Gradient descent is an optimization algorithm ...,Adam optimizer combines momentum and adaptive ...,Yes
2,what helps retrieval when query words do not m...,TF-IDF stands for Term Frequency-Inverse Docum...,TF-IDF stands for Term Frequency-Inverse Docum...,No


## Bonus 1

In [19]:
class WeightedHybridRetriever(HybridRetriever):
    def __init__(self, corpus: List[str], k: int = 60, alpha: float = 0.5):
        super().__init__(corpus, k)
        self.alpha = alpha

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict]:
        bm25_scores = self.bm25.get_scores(tokenize_for_bm25(query))
        bm25_order = np.argsort(bm25_scores)[::-1]
        bm25_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(bm25_order)}

        query_emb = self.embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
        dense_scores = cosine_similarity(query_emb, self.doc_embeddings)[0]
        sbert_order = np.argsort(dense_scores)[::-1]
        sbert_ranks = {int(doc_id): rank + 1 for rank, doc_id in enumerate(sbert_order)}

        fused = []
        for doc_id, text in enumerate(self.corpus):
            bm25_rank = bm25_ranks[doc_id]
            sbert_rank = sbert_ranks[doc_id]
            score = (
                self.alpha * (1.0 / (self.k + bm25_rank)) +
                (1 - self.alpha) * (1.0 / (self.k + sbert_rank))
            )
            fused.append({
                "doc_id": doc_id,
                "rrf_score": float(score),
                "bm25_rank": int(bm25_rank),
                "sbert_rank": int(sbert_rank),
                "text": text
            })

        fused = sorted(fused, key=lambda x: x["rrf_score"], reverse=True)
        return fused[:top_k]

# Example:
weighted = WeightedHybridRetriever(corpus, alpha=0.7)
pretty_print_results(weighted.retrieve("AdamW optimization", top_k=5), "Weighted RRF Example")


Weighted RRF Example
--------------------
1. doc_id=3
   rrf_score   = 0.016314
   bm25_rank   = 1
   sbert_rank  = 2
   text        = Gradient descent is an optimization algorithm used to minimize loss functions.
2. doc_id=9
   rrf_score   = 0.015576
   bm25_rank   = 2
   sbert_rank  = 10
   text        = Tokenization converts raw text into tokens for model input processing.
3. doc_id=7
   rrf_score   = 0.015531
   bm25_rank   = 5
   sbert_rank  = 3
   text        = Regularization techniques like dropout help prevent overfitting in neural networks.
4. doc_id=8
   rrf_score   = 0.015483
   bm25_rank   = 4
   sbert_rank  = 6
   text        = BERT is a transformer-based model pretrained using masked language modeling.
5. doc_id=10
   rrf_score   = 0.015336
   bm25_rank   = 3
   sbert_rank  = 11
   text        = TF-IDF stands for Term Frequency-Inverse Document Frequency and is used in information retrieval.


## Bonus 2

In [20]:
long_doc = """
Transformers are deep learning models that rely heavily on attention mechanisms to process sequential data.
Unlike traditional recurrent neural networks, transformers process all tokens in parallel, allowing for efficient computation.
They use self-attention to capture relationships between words regardless of their distance in the sequence.

Training transformers involves optimizing large numbers of parameters using gradient-based methods such as Adam.
Learning rate scheduling techniques like warmup and decay are often used to stabilize training.
Regularization methods such as dropout and weight decay help prevent overfitting.

Transformers are widely used in natural language processing tasks such as machine translation, text summarization, and question answering.
Models like BERT and GPT are based on transformer architectures and have achieved state-of-the-art performance.

The architecture consists of an encoder and decoder stack, each made up of multiple layers.
Each layer includes multi-head attention and feed-forward neural networks.
Positional encoding is used to inject order information into the model.

Due to their scalability, transformers require large datasets and significant computational resources.
Despite this, they have become the dominant paradigm in modern AI systems.

Recent research focuses on improving efficiency, reducing memory usage, and enhancing interpretability.
"""

In [21]:
def chunk_text(text, chunk_size):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

In [22]:
query = "how are transformers trained?"

for size in [50, 100, 200]:
    chunks = chunk_text(long_doc, size)
    test_corpus = corpus + chunks
    
    temp_retriever = HybridRetriever(test_corpus)
    result = temp_retriever.retrieve(query, top_k=1)
    
    print(f"\nChunk size = {size}")
    print("Top doc:", result[0]["text"])


Chunk size = 50
Top doc: Transformers are deep learning models that rely heavily on attention mechanisms to process sequential data. Unlike traditional recurrent neural networks, transformers process all tokens in parallel, allowing for efficient computation. They use self-attention to capture relationships between words regardless of their distance in the sequence. Training transformers involves optimizing large

Chunk size = 100
Top doc: Transformers use self-attention mechanisms to encode relationships between words in a sequence.

Chunk size = 200
Top doc: Transformers use self-attention mechanisms to encode relationships between words in a sequence.


## Bonus 3

In [23]:
class ColBERTRetriever:
    def __init__(self, corpus):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')
        self.corpus = corpus
        
        self.doc_tokens = [
            self.model.encode(doc.split(), convert_to_tensor=True)
            for doc in corpus
        ]

    def get_scores(self, query):
        query_tokens = self.model.encode(query.split(), convert_to_tensor=True)
        
        scores = []
        
        for doc_emb in self.doc_tokens:
            sim_matrix = util.cos_sim(query_tokens, doc_emb)
            max_sim = sim_matrix.max(dim=1).values.sum().item()
            scores.append(max_sim)
        
        return np.array(scores)

In [24]:
colbert = ColBERTRetriever(corpus)

In [29]:
def hybrid_with_colbert(query, corpus, top_k=5, k=60):
    from rank_bm25 import BM25Okapi
    from sentence_transformers import SentenceTransformer, util
    import numpy as np
    
    # ---- BM25 (fresh) ----
    tokenized_corpus = [doc.lower().split() for doc in corpus]
    bm25 = BM25Okapi(tokenized_corpus)
    
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_ranks = np.argsort(bm25_scores)[::-1]
    
    # ---- SBERT (fresh) ----
    sbert = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = sbert.encode(corpus, convert_to_tensor=True)
    
    query_emb = sbert.encode(query, convert_to_tensor=True)
    sbert_scores = util.cos_sim(query_emb, embeddings)[0].cpu().numpy()
    sbert_ranks = np.argsort(sbert_scores)[::-1]
    
    # ---- ColBERT ----
    colbert = ColBERTRetriever(corpus)
    colbert_scores = colbert.get_scores(query)
    colbert_ranks = np.argsort(colbert_scores)[::-1]
    
    # ---- Combine Ranks ----
    scores = {}
    
    for rank, doc_id in enumerate(bm25_ranks):
        scores.setdefault(doc_id, {"bm25": None, "sbert": None, "colbert": None})
        scores[doc_id]["bm25"] = rank + 1
        
    for rank, doc_id in enumerate(sbert_ranks):
        scores.setdefault(doc_id, {"bm25": None, "sbert": None, "colbert": None})
        scores[doc_id]["sbert"] = rank + 1
        
    for rank, doc_id in enumerate(colbert_ranks):
        scores.setdefault(doc_id, {"bm25": None, "sbert": None, "colbert": None})
        scores[doc_id]["colbert"] = rank + 1
    
    # ---- RRF Fusion ----
    results = []
    
    for doc_id, ranks in scores.items():
        rrf = 0
        
        if ranks["bm25"]:
            rrf += 1 / (k + ranks["bm25"])
        if ranks["sbert"]:
            rrf += 1 / (k + ranks["sbert"])
        if ranks["colbert"]:
            rrf += 1 / (k + ranks["colbert"])
        
        results.append((rrf, corpus[doc_id]))
    
    results.sort(reverse=True)
    return results[:top_k]

In [30]:
query = "transformer attention mechanism"

print("3-Way Hybrid Results:\n")

results = hybrid_with_colbert(query, corpus, top_k=3)

for r in results:
    print("-", r[1])

3-Way Hybrid Results:

- The attention mechanism assigns weights to tokens based on relevance to a query.
- BERT is a transformer-based model pretrained using masked language modeling.
- Transformers use self-attention mechanisms to encode relationships between words in a sequence.
